In [ ]:
import os
import pandas as pd                                         # for data manipulation
import torch                                                # for tensor computations
from transformers import GPT2LMHeadModel, GPT2Tokenizer     # for GPT-2 model and tokenizer
from tqdm import tqdm                                       # for progress bar
import numpy as np                                          # for numerical operations
import nltk                                                 # Natural Language Toolkit
from nltk.tokenize import sent_tokenize                     # for sentence tokenization
import re                                                   # for regex operations
import spacy                                                # for NLP processing
from empath import Empath                                   # for LIWC analysis
from sentence_transformers import SentenceTransformer, util  # for semantic analysis
import collections                                          # for counting duplicates
import random                                               # for random seed setting

# Download necessary NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


SEED = 999
random.seed(SEED)
np.random.seed(SEED)

# Feature Engineering Pipeline

Call `process_dataset()` with your dataset path, output folder, and list of writer columns (include the human column) to run the full feature extraction pipeline.


In [2]:

# ==================== FEATURE EXTRACTION HELPERS ====================

def is_valid_text(text):
    """Check if text is a non-empty string."""
    return isinstance(text, str) and len(text.strip()) > 0


def calculate_perplexity(text, model, tokenizer, device, stride=512):
    """Calculate perplexity using GPT-2 with a sliding window."""
    if not is_valid_text(text):
        return None

    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = int(1e30)
    
    encodings = tokenizer(text, return_tensors="pt", truncation=False)
    tokenizer.model_max_length = original_max_length
    
    seq_len = encodings.input_ids.size(1)
    max_length = model.config.n_positions

    nlls = []
    prev_end_loc = 0

    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            with torch.amp.autocast(device_type=device):
                outputs = model(input_ids, labels=target_ids)
                neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    return torch.exp(torch.stack(nlls).sum() / seq_len).item()


def process_perplexity(texts_list, model, tokenizer, device):
    """Batch perplexity computation."""
    return [calculate_perplexity(text, model, tokenizer, device) for text in tqdm(texts_list, desc="Calculating Perplexity")]


def calculate_burstiness(text):
    """Sentence-length variance."""
    if not is_valid_text(text):
        return 0.0
    sentences = sent_tokenize(text)
    lengths = [len(s.split()) for s in sentences if len(s.split()) > 0]
    if len(lengths) < 2:
        return 0.0
    return float(np.std(lengths))


def process_burstiness_list(texts_list):
    return [calculate_burstiness(text) for text in tqdm(texts_list, desc="Calculating Burstiness")]


def calculate_ttr(text):
    """Type-Token Ratio (lexical diversity)."""
    if not is_valid_text(text):
        return 0.0
    words = re.findall(r"\w+", text.lower())
    if not words:
        return 0.0
    return float(len(set(words)) / len(words))


def process_ttr_list(texts_list):
    return [calculate_ttr(text) for text in tqdm(texts_list, desc="Calculating Lexical Diversity (TTR)")]


def extract_stylometric_features(text):
    """Return [complex_word_ratio, avg_word_len, avg_sentence_len]."""
    if not is_valid_text(text):
        return [0.0, 0.0, 0.0]
    words = re.findall(r"\w+", text.lower())
    sentences = sent_tokenize(text)
    if not words or not sentences:
        return [0.0, 0.0, 0.0]
    avg_word_len = float(np.mean([len(w) for w in words]))
    avg_sent_len = float(len(words) / len(sentences))
    complex_ratio = float(len([w for w in words if len(w) > 6]) / len(words))
    return [complex_ratio, avg_word_len, avg_sent_len]


def calculate_uid(text, model, tokenizer, device, max_length=1024, stride=512):
    """Uniform Information Density (std of surprisal) with sliding window for long texts."""
    if not is_valid_text(text):
        return 0.0
    
    # Temporarily increase model_max_length to suppress warnings
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = int(1e30)
    
    inputs = tokenizer(text, return_tensors="pt", truncation=False)
    tokenizer.model_max_length = original_max_length
    
    input_ids = inputs["input_ids"]
    seq_len = input_ids.size(1)
    
    if seq_len < 2:
        return 0.0
    
    all_surprisals = []
    
    # Sliding window for long texts
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        
        chunk_ids = input_ids[:, begin_loc:end_loc].to(device)
        
        if chunk_ids.shape[1] < 2:
            continue
            
        with torch.no_grad():
            with torch.amp.autocast(device_type=device, enabled=(device == "cuda")):
                outputs = model(chunk_ids, labels=chunk_ids)
                logits = outputs.logits
        
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = chunk_ids[:, 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        surprisal = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        all_surprisals.append(surprisal)
        
        if end_loc >= seq_len:
            break
    
    if not all_surprisals:
        return 0.0
    
    # Concatenate all surprisals and compute std
    combined_surprisals = torch.cat(all_surprisals)
    return float(torch.std(combined_surprisals).item())


def extract_empath(df, text_column, lexicon,categories=None):
    """Empath LIWC features for one text column."""
    print(f"Processing Empath features for: {text_column} (categories: {categories})...")
    feats = []
    for text in tqdm(df[text_column], desc=text_column):
        if is_valid_text(text):
            res = lexicon.analyze(text, categories=categories, normalize=True) or {}
        else:
            res = {}
        res_filled = {cat: res.get(cat, 0.0) for cat in categories}
        feats.append(res_filled)
    temp_df = pd.DataFrame(feats).add_prefix(f"{text_column}_")
    return temp_df


def calculate_avg_syntax_depth(text, nlp):
    """Average parse-tree depth per sentence."""
    if not is_valid_text(text):
        return 0.0
    doc = nlp(text)
    def walk(node, depth):
        if node.n_lefts + node.n_rights == 0:
            return depth
        return max(walk(child, depth + 1) for child in node.children)
    depths = [walk(sent.root, 1) for sent in doc.sents]
    return float(np.mean(depths)) if depths else 0.0


def get_semantic_features(text, semantic_model, device):
    """(mean_cosine, std_cosine) between consecutive sentences."""
    if not is_valid_text(text):
        return 0.0, 0.0
    sentences = [s for s in sent_tokenize(text) if len(s.split()) > 1]
    if len(sentences) < 2:
        return 0.0, 0.0
    with torch.no_grad():
        embeddings = semantic_model.encode(sentences, convert_to_tensor=True, device=device)
    sims = [util.cos_sim(embeddings[i], embeddings[i+1]).item() for i in range(len(embeddings) - 1)]
    if not sims:
        return 0.0, 0.0
    return float(np.mean(sims)), float(np.std(sims))


In [3]:

# ==================== MAIN PIPELINE ====================

def process_dataset(input_dataset: str, output_path: str, writers: list[str], human_col_name: str = 'Human_story') -> pd.DataFrame:
    """Run full feature pipeline for the provided writers.

    input_dataset: CSV path containing at least the writer columns
    output_path: folder to store intermediate and final CSVs
    writers: list of column names to process (include the human column)
    human_col_name: name of the human-written column
    """
    os.makedirs(output_path, exist_ok=True)

    if human_col_name not in writers:
        print(f"Adding missing human column '{human_col_name}' to writers list.")
        writers = [human_col_name] + writers

    print("=" * 80)
    print("FEATURE ENGINEERING PIPELINE - STARTING")
    print("=" * 80)

    np.random.seed(999)
    torch.manual_seed(999)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(999)

    print("[1/10] Loading dataset...")
    base_df = pd.read_csv(input_dataset)
    missing = [w for w in writers if w not in base_df.columns]
    if missing:
        raise ValueError(f"Writer columns not found in dataset: {missing}")
    print(f"Dataset shape: {base_df.shape}")
    print(f"Writers: {writers}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load GPT-2 once for both perplexity and UID to save time/memory
    model_id = 'gpt2'
    tokenizer = GPT2Tokenizer.from_pretrained(model_id)
    model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
    model = model.half() if device == "cuda" else model
    model.eval()

    # ==================== PERPLEXITY ====================
    print("[2/10] Calculating Perplexity...")
    df = base_df.copy()
    for writer in writers:
        df[f"{writer}_ppl"] = process_perplexity(df[writer].tolist(), model, tokenizer, device)
    df.to_csv(os.path.join(output_path, "DB_with_perplexity.csv"), index=False)
    print("Perplexity calculated and saved")

    # ==================== BURSTINESS ====================
    print("[3/10] Calculating Burstiness...")
    df_burst = base_df.copy()
    for writer in writers:
        df_burst[f"{writer}_burstiness"] = process_burstiness_list(df_burst[writer].tolist())
    df_burst.to_csv(os.path.join(output_path, "DB_with_burstiness.csv"), index=False)
    print("Burstiness calculated and saved")

    # ==================== TYPE-TOKEN RATIO (TTR) ====================
    print("[4/10] Calculating Type-Token Ratio (TTR)...")
    df_ttr = base_df.copy()
    for writer in writers:
        df_ttr[f"{writer}_ttr"] = process_ttr_list(df_ttr[writer].tolist())
    df_ttr.to_csv(os.path.join(output_path, "DB_with_TTR.csv"), index=False)
    print("TTR calculated and saved")

    # ==================== STYLOMETRY ====================
    print("[5/10] Calculating Stylometric Features...")
    df_style = base_df.copy()
    for writer in writers:
        feats = df_style[writer].apply(extract_stylometric_features)
        df_style[[f"{writer}_complex", f"{writer}_avg_word_len", f"{writer}_avg_sent_len"]] = pd.DataFrame(feats.tolist(), index=df_style.index)
    df_style.to_csv(os.path.join(output_path, "DB_with_stylometry.csv"), index=False)
    print("Stylometry calculated and saved")

    # ==================== UNIFORM INFORMATION DENSITY (UID) ====================
    print("[6/10] Calculating Uniform Information Density (UID)...")
    df_uid = base_df.copy()
    for writer in writers:
        df_uid[f"{writer}_uid"] = df_uid[writer].apply(lambda x: calculate_uid(x, model, tokenizer, device))
    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    df_uid.to_csv(os.path.join(output_path, "DB_with_uid.csv"), index=False)
    print("UID calculated and saved")

    # ==================== LIWC (EMPATH) ====================
    print("[7/10] Calculating LIWC (Empath) Features...")
    # These categories were selected based analysis of `original_DB.csv` and t-tests
    USED_EMPATH_CATEGORIES = [
        'gain', 'beauty', 'government', 'urban', 'art',
        'help', 'optimism', 'strength', 'love', 'traveling',
    ]

    print(f"Using categories: {USED_EMPATH_CATEGORIES}")
    df_liwc = base_df.copy()
    lexicon = Empath()
    empath_frames = [extract_empath(df_liwc, writer, lexicon, categories=USED_EMPATH_CATEGORIES) for writer in writers]
    df_liwc = pd.concat([df_liwc] + empath_frames, axis=1)

    df_liwc.to_csv(os.path.join(output_path, "DB_with_all_empath.csv"), index=False)
    print("LIWC (Empath) calculated and saved")

    # ==================== SYNTAX TREE DEPTH ====================
    print("[8/10] Calculating Syntax Tree Depth...")
    df_syntax = base_df.copy()
    spacy.prefer_gpu()
    nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    for writer in writers:
        df_syntax[f"{writer}_syntax_depth"] = df_syntax[writer].apply(lambda x: calculate_avg_syntax_depth(x, nlp))
    df_syntax.to_csv(os.path.join(output_path, "DB_with_syntax_depth.csv"), index=False)
    print("Syntax depth calculated and saved")

    # ==================== SEMANTIC CONSISTENCY ====================
    print("[9/10] Calculating Semantic Consistency...")
    df_sem = base_df.copy()
    semantic_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)
    for writer in writers:
        means, stds = [], []
        for text in tqdm(df_sem[writer], desc=writer):
            mean_sim, std_sim = get_semantic_features(text, semantic_model, device)
            means.append(mean_sim)
            stds.append(std_sim)
        df_sem[f"{writer}_semantic_mean"] = means
        df_sem[f"{writer}_semantic_std"] = stds
    df_sem.to_csv(os.path.join(output_path, "DB_with_semantic.csv"), index=False)
    print("Semantic consistency calculated and saved")

    # ==================== DATABASE MERGING ====================
    print("[10/10] Merging all features...")
    df = base_df.copy()
    feature_files = [
        "DB_with_all_empath.csv",
        "DB_with_burstiness.csv",
        "DB_with_perplexity.csv",
        "DB_with_stylometry.csv",
        "DB_with_syntax_depth.csv",
        "DB_with_semantic.csv",
        "DB_with_TTR.csv",
        "DB_with_uid.csv",
    ]
    base_columns = set(writers + ['prompt'])

    for fname in feature_files:
        path = os.path.join(output_path, fname)
        if os.path.exists(path):
            print(f"  Merging features from: {fname}")
            temp_df = pd.read_csv(path)
            new_cols = [c for c in temp_df.columns if c not in base_columns]
            df = pd.concat([df, temp_df[new_cols]], axis=1)

    duplicates = [item for item, count in collections.Counter(df.columns).items() if count > 1]
    if duplicates:
        print(f"Warning: Duplicate columns found: {duplicates}")


    # Feature suffixes for final dataframe - use the predefined categories
    feature_suffixes = [
        'ppl','burstiness', 'syntax_depth', 'ttr', 'semantic_mean', 'semantic_std', 'uid'
    ] + USED_EMPATH_CATEGORIES
    print(f"Using feature suffixes: {feature_suffixes}")

    records = []
    for writer in writers:
        is_ai = 0 if writer.lower() == human_col_name.lower() else 1
        for _, row in df.iterrows():
            rec = {'Text': row[writer], 'Writer': writer, 'is_AI': is_ai}
            for feat in feature_suffixes:
                col_name = f"{writer}_{feat}"
                rec[feat] = row[col_name] if col_name in df.columns else None
            records.append(rec)

    train_df = pd.DataFrame(records)
    print(f"Rows before dropna: {len(train_df)}")
    train_df.dropna(inplace=True)
    print(f"Rows after dropna: {len(train_df)}")

    train_df.to_csv(os.path.join(output_path, "DB_final.csv"), index=False)
    df.to_csv(os.path.join(output_path, "DB_merged.csv"), index=False)

    print("" + "=" * 80)
    print("PIPELINE COMPLETE!")
    print("=" * 80)
    print(f"Final DataFrame Shape: {train_df.shape}")
    print(f"Total Samples: {len(train_df)}")
    print(f"All files saved to: {output_path}")
    print("=" * 80)
    return train_df


## Usage

In [ ]:
# input_files=[
#     "data/original_DB.csv"
#  "data/misspelled/spelling_errors_test_set.csv",
#  "data/MTG_Benchmark/unseen/Reuters_LLMs_test.csv",
#  "data/MTG_Benchmark/Cross-Domain/Essay_LLMs_test.csv",
#  "data/MTG_Benchmark/Cross-Domain/WP_LLMs_test.csv",
#  "data/paraphrasing/nllb/DB_test_paraphrased_1.3B_FINAL.csv",
#  "data/paraphrasing/gemini/DB_paraphrased_500_FINAL.csv"
# ]
# output_files=[
#      "data/features"
#      "data/misspelled/",
#     "data/MTG_Benchmark/unseen/",
#     "data/MTG_Benchmark/Cross-Domain/Essay/",   # separate folder
#     "data/MTG_Benchmark/Cross-Domain/WP/",       # separate folder
#     "data/paraphrasing/nllb/",
#     "data/paraphrasing/gemini/"
# ]
# writers=[
#     ['Human_story', 'gemma-2-9b', 'mistral-7B', 'qwen-2-72B', 'llama-8B',
#        'accounts/yi-01-ai/models/yi-large', 'GPT_4-o'],
#     ['Human_story', 'gemma-2-9b', 'mistral-7B', 'qwen-2-72B', 'llama-8B',
#        'accounts/yi-01-ai/models/yi-large', 'GPT_4-o'],
#     ["Claude","human"],
#     ["GPT4All","human"],
#     ["GPT4All","human"],
#     ['Human_story_translated', 'gemma-2-9b_translated', 'mistral-7B_translated', 'qwen-2-72B_translated', 'llama-8B_translated',
#        'yi-large_translated', 'GPT_4-o_translated']
#     ['Human_story_paraphrased', 'gemma-2-9b_paraphrased', 'mistral-7B_paraphrased', 'qwen-2-72B_paraphrased', 'llama-8B_paraphrased',
#        'yi-large_paraphrased', 'GPT_4-o_paraphrased']
    
# ]
# human_col_name=[
#      "Human_story",
#      "Human_story",
#      "human",
#      "human",
#      "human",
#      Human_story_translated"
#      "Human_story_paraphrased"
# ]

# for i in range(len(input_files)):
#     process_dataset(input_files[i], output_files[i], writers[i], human_col_name[i])
